<h1>IMPORTAÇÃO DE BIBLIOTECAS</h1>

In [3]:
import pandas as pd
import numpy as np

import os
import shutil


<H2> A) Segmentação dos dados <H2>

In [4]:
preprocessing_path = "./preprocessing/"
path_segments = preprocessing_path + "segments/"
deltaT = 10000

original_dataset_filename = "02-raw_dataset.csv"
segmented_dataset_fileName = "03-segmented_dataset.csv"

def limparPasta(file):
    for arquivo in os.listdir(file):
        caminho = os.path.join(file, arquivo)
        if os.path.isfile(caminho) or os.path.islink(caminho):
            os.remove(caminho)
        elif os.path.isdir(caminho):
            shutil.rmtree(caminho)

def load_rawData(dataFile):
    df_rawData = pd.read_csv(dataFile, sep=",",decimal=".") # lendo arquivo

    df_rawData["module"] = np.sqrt(np.pow(df_rawData["Q_data"],2) + np.pow(df_rawData["I_data"],2)) # Calculando modulo do sinal
    df_rawData["fase"] = np.atan(df_rawData["Q_data"]/df_rawData["I_data"]) # Calculando fase do sinal

    df_rawData = df_rawData[df_rawData["module"] > 0] # Filtrando valores não nulos

    return df_rawData

def data_segment(data, deltaT):
    data = data.copy()

    data_segments = []
    rawData = load_rawData(dataFile=data["rawData"])
    indice = 0
    for i in range(0, rawData.shape[0], deltaT):
        newData = data.to_dict()
        newData['dataName'] = newData['dataName'] + "-"+ str(indice)
        indice = indice + 1
        newData['rawData'] = path_segments + newData['dataName'] + ".csv"
        rawData[["Q_data", "I_data"]].loc[i:i+deltaT].to_csv(newData['rawData'], index=False)
        data_segments.append(newData)
    return data_segments

df_dataDetails = pd.read_csv(preprocessing_path + original_dataset_filename)

print("Limpando pasta de segmentos....")
limparPasta(path_segments)

print("Iniciando processo de segmentação....")
datas_segments_list = []
total = df_dataDetails.shape[0]
for indice, data in df_dataDetails.iterrows():
    data_segments = data_segment(data, deltaT)
    datas_segments_list = datas_segments_list + data_segments
    print(f"\r{((indice + 1) / total) * 100:.2f}% concluído!", end="")
datas_segments = pd.DataFrame(datas_segments_list)

datas_segments.to_csv(preprocessing_path + segmented_dataset_fileName, index=False)
datas_segments.head()
        

Limpando pasta de segmentos....
Iniciando processo de segmentação....
100.00% concluído!

,dataName,motorSpeed,load,class,rawData
0,data1-0,500.0,0.0,Normal,./preprocessing/segments/data1-0.csv
1,data1-1,500.0,0.0,Normal,./preprocessing/segments/data1-1.csv
2,data1-2,500.0,0.0,Normal,./preprocessing/segments/data1-2.csv
3,data1-3,500.0,0.0,Normal,./preprocessing/segments/data1-3.csv
4,data1-4,500.0,0.0,Normal,./preprocessing/segments/data1-4.csv
